# 06 — Explainability (SHAP and honest caveats)

## Feature importance vs SHAP

**Tree feature importance** (gain/weight) is **global** and can be biased toward high-cardinality splits.

**SHAP** (SHapley Additive exPlanations):

1. **What:** attributes a prediction to features using a game-theoretic average of marginal contributions.
2. **Why:** reviewers ask "why did *this* payment look like fraud?"
3. **How:** TreeSHAP computes exact Shapley values for tree ensembles efficiently.
4. **Here:** global summary on a sample + one local explanation.

## Hard rule

SHAP tells you **what the model used**. It does **not** prove that a PCA component *caused* fraud. Correlation is not causation; attribution is not causation either.

V1–V28 are anonymized. Saying "V14 caused the decline" would be scientifically false.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_processing import dataset_overview, drop_duplicates_if_present, load_raw_dataset
from src.utils import RAW_DATASET_PATH

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.predict import load_artifacts, predict_risk
from src.data_processing import stratified_train_test_split

artifacts = load_artifacts()
meta = artifacts["metadata"]
print("model:", meta.get("model_name"))
df = drop_duplicates_if_present(load_raw_dataset())
X_train, X_test, y_train, y_test = stratified_train_test_split(df)


## Local explanation for one test row (app-style contributions)


In [ ]:
row = X_test.iloc[0]
result = predict_risk(row, artifacts=artifacts)
print("probability", result["fraud_probability"])
print("score", result["risk_score"], result["risk_category"])
pd.DataFrame(result["contributing_features"])


## Global SHAP (sample)

If this cell fails (install issues), the application still uses the contribution fallback in `predict.py`. That is intentional: the product should not hard-crash without SHAP.


In [ ]:
try:
    import shap
    pipe = artifacts["pipeline"]
    model = pipe.named_steps["model"]
    pre = pipe.named_steps["preprocess"]
    sample = X_test.sample(n=min(300, len(X_test)), random_state=42)
    Xt = pre.transform(sample)
    names = list(pre.get_feature_names_out())
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(Xt)
    if isinstance(sv, list):
        sv = sv[1]
    shap.summary_plot(sv, Xt, feature_names=names, show=True)
except Exception as exc:
    print("SHAP skipped:", type(exc).__name__, exc)
